<a href="https://colab.research.google.com/github/csukuangfj/kaldi-hmm-gmm/blob/master/mfa_chinese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This colab notebook shows how to use kaldi-hmm-gmm for forced alignment for Chinese.

See also
https://csukuangfj.github.io/kaldi-hmm-gmm/usage/mfa-mandarin.html


# Environment setup

## Install dependencies

In [1]:
! pip install kaldi-hmm-gmm kaldi-native-fbank kaldifst numpy soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kaldi-hmm-gmm]


## Download source code

In [2]:
%%shell

git clone https://github.com/csukuangfj/kaldi-hmm-gmm
cd kaldi-hmm-gmm

# 2. download the model (the dictionary ships inside the zip) and the audio
mkdir -p aishell3_pinyin mandarin && cd aishell3_pinyin
wget https://github.com/lars76/forced-alignment-chinese/releases/download/model_files/aishell3_pinyin.zip
unzip aishell3_pinyin.zip          # -> aishell3_pinyin_acoustic.zip + dictionaries
unzip aishell3_pinyin_acoustic.zip # -> aishell3_pinyin_acoustic/
cd ..
wget -O mandarin/zh.wav \
  https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/zh.wav


Cloning into 'kaldi-hmm-gmm'...
remote: Enumerating objects: 1855, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (291/291), done.
remote: Total 1855 (delta 195), reused 116 (delta 108), pack-reused 1455 (from 3)
Receiving objects: 100% (1855/1855), 9.61 MiB | 23.31 MiB/s, done.
Resolving deltas: 100% (785/785), done.
--2026-11-11 01:42:17--  https://github.com/lars76/forced-alignment-chinese/releases/download/model_files/aishell3_pinyin.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/812381970/a215e18a-a125-4b22-81b0-0868e1415981?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-11T02%3A15%3A44Z&rscd=attachment%3B+filename%3Daishell3_pinyin.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47

In [3]:
%%shell

ls -lh kaldi-hmm-gmm/aishell3_pinyin/

total 117M
drwxr-xr-x 2 root root 4.0K Aug  3  2024 aishell3_pinyin_acoustic
-rw-r--r-- 1 root root  59M Aug  3  2024 aishell3_pinyin_acoustic.zip
-rw-r--r-- 1 root root  20K Aug  3  2024 aishell3_pinyin_dictionary.dict
-rw-r--r-- 1 root root 7.8K Aug  2  2024 aishell3_pinyin_dictionary.txt
-rw-r--r-- 1 root root  59M Aug  3  2024 aishell3_pinyin.zip


In [4]:
%%shell

ls -lh kaldi-hmm-gmm/aishell3_pinyin/aishell3_pinyin_acoustic

total 65M
-rw-r--r-- 1 root root  32M Aug  3  2024 final.alimdl
-rw-r--r-- 1 root root  32M Aug  3  2024 final.mdl
-rw-r--r-- 1 root root  247 Aug  3  2024 graphemes.txt
-rw-r--r-- 1 root root  15K Aug  3  2024 lda.mat
-rw-r--r-- 1 root root 3.2K Aug  3  2024 meta.json
-rw-r--r-- 1 root root 472K Aug  3  2024 phone_lm.fst
-rw-r--r-- 1 root root  66K Aug  3  2024 phone_pdf.counts
-rw-r--r-- 1 root root  361 Aug  3  2024 phones.txt
-rw-r--r-- 1 root root 225K Aug  3  2024 tree


In [5]:
%%shell

head kaldi-hmm-gmm/aishell3_pinyin/aishell3_pinyin_dictionary.dict

<cutoff>	0.99	0.17	1.0	1.0	spn
<unk>	0.99	0.17	1.0	1.0	spn
[bracketed]	0.99	0.17	1.0	1.0	spn
[laughter]	0.99	0.17	1.0	1.0	spn
a	0.99	0.15	5.17	0.48	a
ai	0.99	0.15	3.33	0.71	ai̯
an	0.99	0.14	2.35	0.82	a n
ang	0.99	0.13	1.81	0.8	a ŋ
ao	0.99	0.05	3.32	0.69	au̯
ba	0.99	0.21	1.62	0.91	p a


In [6]:
%%shell

head kaldi-hmm-gmm/aishell3_pinyin/aishell3_pinyin_dictionary.txt

a	a
ai	ai̯
an	a n
ang	a ŋ
ao	au̯
ba	p a
bai	p ai̯
ban	p a n
bang	p a ŋ
bangr	p ɑ̃ ɻ


In [7]:
%%shell

ls -lh kaldi-hmm-gmm/mandarin

total 224K
-rw-r--r-- 1 root root 224K Feb 27  2026 zh.wav


In [8]:
from IPython.display import Audio, display

sound_file = "./kaldi-hmm-gmm/mandarin/zh.wav"
display(Audio(sound_file, autoplay=True))

## Do it

In [9]:
%%shell

pip install pypinyin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 5.1 MB/s  0:00:00


In [10]:
%%shell

cd kaldi-hmm-gmm


python3 tools/align_mfa.py --wav mandarin/zh.wav \
  --text "不要问你的国家能为你做什么，而要问你能为你的国家做什么。" \
  --model-dir aishell3_pinyin/aishell3_pinyin_acoustic \
  --dict aishell3_pinyin/aishell3_pinyin_dictionary.dict \
  --phones-txt aishell3_pinyin/aishell3_pinyin_acoustic/phones.txt \
  --tokenize pinyin --out mandarin/zh-aishell3.TextGrid

OK: mandarin/zh-aishell3.TextGrid  frames=476 dur=4.76s phones=71 words=26
timing: audio duration 4.76s, elapsed 25.52s, RTF 5.363


## Show the result

In [11]:
! cat kaldi-hmm-gmm/mandarin/zh-aishell3.TextGrid

File type = "ooTextFile"
Object class = "TextGrid"

xmin = 0
xmax = 4.7600
tiers? <exists>
size = 2
item []:
    item [1]:
        class = "IntervalTier"
        name = "phones"
        xmin = 0
        xmax = 4.7600
        intervals: size = 71
        intervals [1]:
            xmin = 0.0000
            xmax = 0.0600
            text = "sil"
        intervals [2]:
            xmin = 0.0600
            xmax = 0.1200
            text = "p"
        intervals [3]:
            xmin = 0.1200
            xmax = 0.1800
            text = "u"
        intervals [4]:
            xmin = 0.1800
            xmax = 0.2700
            text = "j"
        intervals [5]:
            xmin = 0.2700
            xmax = 0.3200
            text = "au̯"
        intervals [6]:
            xmin = 0.3200
            xmax = 0.3900
            text = "w"
        intervals [7]:
            xmin = 0.3900
            xmax = 0.4200
            text = "ə"
        intervals [8]:
            xmin = 0.4200
            xma

## Extract audio segment for each word

In [12]:
%%shell
cd kaldi-hmm-gmm

python3 tools/textgrid_to_word_wavs.py --wav mandarin/zh.wav \
  --textgrid mandarin/zh-aishell3.TextGrid --out-dir mandarin/zh-aishell3.words

w01.wav  0.06-0.18s  不
w02.wav  0.18-0.32s  要
w03.wav  0.32-0.48s  问
w04.wav  0.48-0.64s  你
w05.wav  0.64-0.76s  的
w06.wav  0.76-0.95s  国
w07.wav  0.99-1.14s  家
w08.wav  1.14-1.32s  能
w09.wav  1.32-1.44s  为
w10.wav  1.44-1.60s  你
w11.wav  1.63-1.73s  做
w12.wav  1.76-1.94s  什
w13.wav  1.94-2.19s  么
w14.wav  2.48-2.55s  而
w15.wav  2.55-2.74s  要
w16.wav  2.74-2.89s  问
w17.wav  2.89-3.04s  你
w18.wav  3.04-3.20s  能
w19.wav  3.20-3.33s  为
w20.wav  3.33-3.50s  你
w21.wav  3.50-3.61s  的
w22.wav  3.61-3.77s  国
w23.wav  3.77-3.95s  家
w24.wav  3.99-4.11s  做
w25.wav  4.14-4.34s  什
w26.wav  4.34-4.55s  么
wrote 26 clips + manifest.json to mandarin/zh-aishell3.words/


In [13]:
%%shell

ls -lh kaldi-hmm-gmm/mandarin/zh-aishell3.words/

total 244K
-rw-r--r-- 1 root root 2.2K Nov 11 01:42 manifest.json
-rw-r--r-- 1 root root 5.7K Nov 11 01:42 w01.wav
-rw-r--r-- 1 root root 6.7K Nov 11 01:42 w02.wav
-rw-r--r-- 1 root root 7.6K Nov 11 01:42 w03.wav
-rw-r--r-- 1 root root 7.6K Nov 11 01:42 w04.wav
-rw-r--r-- 1 root root 5.7K Nov 11 01:42 w05.wav
-rw-r--r-- 1 root root 9.0K Nov 11 01:42 w06.wav
-rw-r--r-- 1 root root 7.1K Nov 11 01:42 w07.wav
-rw-r--r-- 1 root root 8.5K Nov 11 01:42 w08.wav
-rw-r--r-- 1 root root 5.7K Nov 11 01:42 w09.wav
-rw-r--r-- 1 root root 7.6K Nov 11 01:42 w10.wav
-rw-r--r-- 1 root root 4.8K Nov 11 01:42 w11.wav
-rw-r--r-- 1 root root 8.5K Nov 11 01:42 w12.wav
-rw-r--r-- 1 root root  12K Nov 11 01:42 w13.wav
-rw-r--r-- 1 root root 3.4K Nov 11 01:42 w14.wav
-rw-r--r-- 1 root root 9.0K Nov 11 01:42 w15.wav
-rw-r--r-- 1 root root 7.1K Nov 11 01:42 w16.wav
-rw-r--r-- 1 root root 7.1K Nov 11 01:42 w17.wav
-rw-r--r-- 1 root root 7.6K Nov 11 01:42 w18.wav
-rw-r--r-- 1 root root 6.2K Nov 11 01:42 w19.wav
-rw

In [14]:
%%shell

cat kaldi-hmm-gmm/mandarin/zh-aishell3.words/manifest.json

[
  {
    "word": "不",
    "start": 0.06,
    "end": 0.18,
    "file": "w01.wav"
  },
  {
    "word": "要",
    "start": 0.18,
    "end": 0.32,
    "file": "w02.wav"
  },
  {
    "word": "问",
    "start": 0.32,
    "end": 0.48,
    "file": "w03.wav"
  },
  {
    "word": "你",
    "start": 0.48,
    "end": 0.64,
    "file": "w04.wav"
  },
  {
    "word": "的",
    "start": 0.64,
    "end": 0.76,
    "file": "w05.wav"
  },
  {
    "word": "国",
    "start": 0.76,
    "end": 0.95,
    "file": "w06.wav"
  },
  {
    "word": "家",
    "start": 0.99,
    "end": 1.14,
    "file": "w07.wav"
  },
  {
    "word": "能",
    "start": 1.14,
    "end": 1.32,
    "file": "w08.wav"
  },
  {
    "word": "为",
    "start": 1.32,
    "end": 1.44,
    "file": "w09.wav"
  },
  {
    "word": "你",
    "start": 1.44,
    "end": 1.6,
    "file": "w10.wav"
  },
  {
    "word": "做",
    "start": 1.63,
    "end": 1.73,
    "file": "w11.wav"
  },
  {
    "word": "什",
    "start": 1.76,
    "end": 1.94,
    "file": "w12.w

## Play it

In [15]:
import json
import os
from IPython.display import Audio, display
import pandas as pd

# Define path constants for Mandarin dataset
base_dir = "kaldi-hmm-gmm/mandarin/zh-aishell3.words"
json_path = os.path.join(base_dir, "manifest.json")

# 1. Load manifest.json
with open(json_path, "r", encoding="utf-8") as f:
  data = json.load(f)

# 2. Convert to DataFrame for structured table display
df = pd.DataFrame(data)
# Calculate duration for each word segment
df["duration"] = (df["end"] - df["start"]).round(2)

print("=== Mandarin Word Intervals Summary Table ===")
display(df)

print("\n=== Mandarin Word Audio Players ===")
# 3. Iterate over each entry to display details and audio player
for index, row in df.iterrows():
  word = row["word"]
  start = row["start"]
  end = row["end"]
  audio_file = row["file"]
  audio_path = os.path.join(base_dir, audio_file)

  print(
      f"Index {index + 1}: Word [{word}] | Interval: {start}s - {end}s"
      f" (Duration: {row['duration']}s) | File: {audio_file}"
  )

  # Render HTML5 audio player widget in Colab
  display(Audio(audio_path))

=== Mandarin Word Intervals Summary Table ===


,word,start,end,file,duration
0,不,0.06,0.18,w01.wav,0.12
1,要,0.18,0.32,w02.wav,0.14
2,问,0.32,0.48,w03.wav,0.16
3,你,0.48,0.64,w04.wav,0.16
4,的,0.64,0.76,w05.wav,0.12
5,国,0.76,0.95,w06.wav,0.19
6,家,0.99,1.14,w07.wav,0.15
7,能,1.14,1.32,w08.wav,0.18
8,为,1.32,1.44,w09.wav,0.12
9,你,1.44,1.60,w10.wav,0.16



=== Mandarin Word Audio Players ===
Index 1: Word [不] | Interval: 0.06s - 0.18s (Duration: 0.12s) | File: w01.wav


Index 2: Word [要] | Interval: 0.18s - 0.32s (Duration: 0.14s) | File: w02.wav


Index 3: Word [问] | Interval: 0.32s - 0.48s (Duration: 0.16s) | File: w03.wav


Index 4: Word [你] | Interval: 0.48s - 0.64s (Duration: 0.16s) | File: w04.wav


... (18 outputs left out)

Index 23: Word [家] | Interval: 3.77s - 3.95s (Duration: 0.18s) | File: w23.wav


Index 24: Word [做] | Interval: 3.99s - 4.11s (Duration: 0.12s) | File: w24.wav


Index 25: Word [什] | Interval: 4.14s - 4.34s (Duration: 0.2s) | File: w25.wav


Index 26: Word [么] | Interval: 4.34s - 4.55s (Duration: 0.21s) | File: w26.wav
